# EMMa Single Material Subset

In [ ]:
import json
from pathlib import Path
import numpy as np

from emma import EmmaDataset, EmmaSearcher

In [ ]:
DATA_DIR = Path().resolve().parent.parent
DATA_CONFIG = DATA_DIR / "configs" / "data.json"
with DATA_CONFIG.open(encoding="utf-8") as f:
    data_config = json.load(f)

DATASET_DIR = Path(data_config["emma_root"])
INPUT_ASINS = Path(data_config["asins"]["post_augmentation"])
SINGLE_MATERIAL_ASINS_PATH = Path(data_config["single_material_asins"])
SINGLE_MATERIAL_MATERIALS_PATH = Path(data_config["single_material_materials"])

In [ ]:
ds = EmmaDataset(DATASET_DIR)

material_names = ds.material_names
hierarchy = ds.material_hierarchy

children = hierarchy["children"]
roots = hierarchy["roots"]
assert material_names == hierarchy["nodes"]
material_idx = {name: i for i, name in enumerate(material_names)}

In [ ]:
def subtree_best(node, probs, memo):
    if node in memo:
        return memo[node]

    best_prob = float(probs[material_idx[node]])
    best_material = node

    for child in children[node]:
        child_prob, child_material = subtree_best(child, probs, memo)

        if child_prob > best_prob:
            best_prob = child_prob
            best_material = child_material

    memo[node] = (best_prob, best_material)
    return memo[node]


def get_single_material(
    probs,
    min_support=0.90,
    max_conflict=0.10,
    min_margin=0.80,
    filter_excluded=False,
):
    probs = np.asarray(probs)
    memo = {}

    root_scores = []

    for root in roots:
        score, best_material = subtree_best(root, probs, memo)
        root_scores.append((score, root, best_material))

    root_scores.sort(reverse=True)

    _, current, _ = root_scores[0]

    if len(root_scores) > 1:
        conflict = root_scores[1][0]
    else:
        conflict = 0.0

    while children[current]:
        branch_scores = []

        for child in children[current]:
            score, best_material = subtree_best(child, probs, memo)
            branch_scores.append((score, child, best_material))

        branch_scores.sort(reverse=True)

        if len(branch_scores) > 1:
            second_score = branch_scores[1][0]
            conflict = max(conflict, second_score)

        best_score, best_child, _ = branch_scores[0]

        if best_score < min_support:
            break

        current = best_child

    material = current
    support = float(probs[material_idx[material]])

    accepted = (
        support >= min_support
        and conflict <= max_conflict
        and support - conflict >= min_margin
    )

    return material if accepted else None

In [ ]:
asins = np.load(INPUT_ASINS, allow_pickle=False)

In [ ]:
selected_asins = []
selected_materials = []

for asin in asins:
    record = ds.get_record(asin)
    material = get_single_material(ds.get_material_probs(record), filter_excluded=False)

    if material is not None:
        selected_asins.append(asin)
        selected_materials.append(material)

print(
    f"{len(selected_asins)} / {len(asins)} "
    "accepted as single-material"
)

In [ ]:
matches = []

for asin, material in zip(selected_asins, selected_materials):
    row = ds.asin_to_index[asin]

    material_target = np.asarray(
        ds.material_bits[row],
        dtype=np.float32,
    )[:182].reshape(-1)

    idx = material_idx[material]
    is_target = material_target[idx] == 1

    matches.append(is_target)

print(f"Accepted: {len(matches)}")
print(f"Returned material is a target: {sum(matches)}")
print(f"Not a target: {len(matches) - sum(matches)}")
print(f"Accuracy among accepted: {np.mean(matches):.2%}")

In [ ]:
selected_asins = np.asarray(selected_asins, dtype=str)
selected_materials = np.asarray(selected_materials, dtype=str)

SINGLE_MATERIAL_ASINS_PATH.parent.mkdir(parents=True, exist_ok=True)
SINGLE_MATERIAL_MATERIALS_PATH.parent.mkdir(parents=True, exist_ok=True)
np.save(SINGLE_MATERIAL_ASINS_PATH, selected_asins)
np.save(SINGLE_MATERIAL_MATERIALS_PATH, selected_materials)